# §2 Reading input: the tokenizer

*Reads `calc/tokens.py`. Refines the fragment ⟨Turn the string into tokens⟩ from §1.*

The first stage answers a narrow question: what are the indivisible *words* of an expression? The tokenizer chops the raw string into a list of **tokens** — a number, an operator, a parenthesis — and throws away whitespace. Crucially it does **not** know that `*` binds tighter than `+`, or that parentheses group. That knowledge belongs to the parser (§3); keeping it out here is what makes both stages simple.

## §2.1 A token, and the kinds there are

A token is just a *kind* tag plus the original text it came from. The kinds are plain string constants — the parser will compare against them by name.

📄 [calc/tokens.py lines 11–35](../sample_repo/calc/tokens.py)

```python
# calc/tokens.py : lines 11-35
# Token kinds (used as opaque tags by the parser).
NUMBER = "NUMBER"
PLUS = "PLUS"
MINUS = "MINUS"
STAR = "STAR"
SLASH = "SLASH"
LPAREN = "LPAREN"
RPAREN = "RPAREN"
EOF = "EOF"

# Single-character operators and punctuation map straight to a kind.
_SINGLE = {
    "+": PLUS,
    "-": MINUS,
    "*": STAR,
    "/": SLASH,
    "(": LPAREN,
    ")": RPAREN,
}


@dataclass
class Token:
    kind: str
    value: str
```

Using bare strings as kinds (rather than an `Enum`) keeps the demo short; a larger language would reach for an enum to get exhaustiveness checking. That is the kind of *alternative* a literate reading should name even when the code does not take it.

## §2.2 The scan loop

Tokenizing is a single left-to-right pass with a manual cursor `i`. At each position one of four things is true, so the loop body is four cases:

📄 [calc/tokens.py lines 38–58](../sample_repo/calc/tokens.py)

```python
# calc/tokens.py : lines 38-58  (skeleton)
def tokenize(text: str) -> list[Token]:
    tokens, i, n = [], 0, len(text)
    while i < n:
        c = text[i]
        ⟨Skip whitespace 2.2.1⟩
        ⟨Emit a single-character operator or paren 2.2.2⟩
        ⟨Read a multi-digit number 2.2.3⟩
        ⟨Otherwise, reject the character 2.2.4⟩
    tokens.append(Token(EOF, ''))   # a sentinel the parser relies on
    return tokens
```

The trailing `EOF` token is worth pausing on: it is a sentinel so the parser can always `peek()` one more token without bounds-checking. A small lie told here buys simplicity downstream — a recurring theme.

### §2.2.1–§2.2.3 The cases that consume input

Whitespace advances the cursor and produces nothing. A single character that appears in the `_SINGLE` table becomes one token. Digits (and `.`) are gathered greedily into one `NUMBER` token — this inner `while` is the only place the tokenizer looks at more than one character at a time:

📄 [calc/tokens.py lines 43–55](../sample_repo/calc/tokens.py)

```python
# calc/tokens.py : lines 43-55
        if c.isspace():
            i += 1
            continue
        if c in _SINGLE:
            tokens.append(Token(_SINGLE[c], c))
            i += 1
            continue
        if c.isdigit() or c == ".":
            start = i
            while i < n and (text[i].isdigit() or text[i] == "."):
                i += 1
            tokens.append(Token(NUMBER, text[start:i]))
            continue
```

Note what the number case does *not* do: it does not convert to `float`. It keeps the raw text and lets the parser decide when to parse the value (§3.5). Each stage does the least it can.

### §2.2.4 Rejecting the unknown

Anything else is a lexical error, reported with its position. This is the tokenizer's only failure mode — and it is caught far away, at the command line's error boundary (§1.1).

📄 [calc/tokens.py line 56](../sample_repo/calc/tokens.py)

```python
# calc/tokens.py : lines 56-56
        raise SyntaxError(f"unexpected character {c!r} at position {i}")
```

## See it run

Watch a string become tokens:

In [1]:
import os, sys
# Find the bundled demo package (works when run from the calc-literate directory).
d = os.getcwd()
for _ in range(6):
    for cand in (os.path.join(d, 'sample_repo'), os.path.join(d, '..', 'sample_repo')):
        if os.path.isdir(os.path.join(cand, 'calc')):
            sys.path.insert(0, os.path.abspath(cand))
            break
    else:
        d = os.path.dirname(d); continue
    break
import calc
print('imported calc', calc.__version__)

imported calc 0.1.0


In [2]:
from calc.tokens import tokenize
for t in tokenize('1 + 2 * 3'):
    print(f'{t.kind:<7} {t.value!r}')

NUMBER  '1'
PLUS    '+'
NUMBER  '2'
STAR    '*'
NUMBER  '3'
EOF     ''


## Where this leads

We now have a flat list of tokens with no structure beyond their order. Giving that list *structure* — discovering that `2 * 3` is a unit inside `1 + 2 * 3` — is the parser's job, and the subject of §3.